In [1]:
!pip list

Package                   Version
------------------------- --------------
accelerate                1.10.1
adapters                  1.1.0
aiohappyeyeballs          2.6.1
aiohttp                   3.11.15
aiosignal                 1.3.2
amdsmi                    24.6.3+52b3947
annotated-types           0.7.0
ansible-core              2.14.18
anyio                     4.7.0
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.4
async-timeout             5.0.1
attrs                     24.3.0
babel                     2.16.0
beautifulsoup4            4.12.3
bleach                    6.2.0
blinker                   1.9.0
Bottleneck                1.5.0
certifi                   2024.12.14
cffi                      1.17.1
chardet                   4.0.0
charset-normalizer        3.4.0
clang                     20.1.5
click                     8.1.8
cockpit                   334.1
col

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from src.models.qwen3 import Qwen3

t = Qwen3(model_name = "Qwen/Qwen3-4B", device = "cuda:0")

t.model

/home1/pedrobpio/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:root:Using device: cuda:0
`torch_dtype` is deprecated! Use `dtype` instead!

oading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.19it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [5]:
from src.datasets.lener import LenerDataset

l = LenerDataset(tokenizer=t.tokenizer)

INFO:datasets:PyTorch version 2.5.1+rocm6.2 available.


In [6]:
data = l.load_dataset()
data

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
INFO:src.datasets.lener:Dataset loaded with splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Processing splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Original columns to remove after mapping: ['id', 'tokens', 'ner_tags']
INFO:src.datasets.lener:NER tag mapping created: {0: 'O', 1: 'B-ORGANIZACAO', 2: 'I-ORGANIZACAO', 3: 'B-PESSOA', 4: 'I-PESSOA', 5: 'B-TEMPO', 6: 'I-TEMPO', 7: 'B-LOCAL', 8: 'I-LOCAL', 9: 'B-LEGISLACAO', 10: 'I-LEGISLACAO', 11: 'B-JURISPRUDENCIA', 12: 'I-JURISPRUDENCIA'}
INFO:src.datasets.lener:Starting dataset mapping...



INFO:src.datasets.lener:Dataset mapping finished.
INFO:src.datasets.lener:Columns after mapping: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels']


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1390
    })
})

In [7]:
from src.peft_configs.lora import LoraAdapter
# PRESET = 'baseline'
PRESET = 'full_attention'
# PRESET = 'all'
# PRESET = 'ffn'
# PRESET = 'full_attention_plus_ffn'
# PRESET = 'low_rank'
# PRESET = 'high_rank'
# PRESET = 'high_rank_XX'

lora = LoraAdapter(
    model=t.model,
    lora_preset = PRESET
)

model = lora.apply_lora()
model

INFO:root:Applying LoRA with preset: full_attention
INFO:root:LoRA configurations: {'r': 8, 'lora_alpha': 16, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lora_dropout': 0.1, 'bias': 'none', 'task_type': <TaskType.CAUSAL_LM: 'CAUSAL_LM'>}


trainable params: 5,898,240 || all params: 4,028,366,336 || trainable%: 0.1464


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch
from trl import DataCollatorForCompletionOnlyLM
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "7"

# Training args
training_args = TrainingArguments(
    output_dir=f'./outputs/checkpoints/Qwen3_4b_full_{PRESET}',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    # gradient_accumulation_steps=4,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="no",
    learning_rate=2e-4,
    warmup_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    # device=model.device
    
)
response_template = "Resposta:\n"

response_template_ids = t.tokenizer.encode(
    response_template, 
    add_special_tokens=False
)

data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=t.tokenizer
)
# model.to('cuda:7')
# torch.cuda.set_device(7)
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"],
    tokenizer=t.tokenizer,
    data_collator=data_collator,
    
)

# Train
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/tmp/ipykernel_1489641/4130380506.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,0.206900
100,0.036900
150,0.022500
200,0.014500
250,0.020600
300,0.011500
350,0.012500
400,0.009900
450,0.009800
500,0.010300


/home1/pedrobpio/.local/lib/python3.9/site-packages/trl/trainer/utils.py:153: UserWarning: Could not find response key `[1061, 38531, 510]` in the following instance: Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto
Texto: Acórdão VISTOS , relatados e discutidos estes autos de denúncia sobre irregularidades

TrainOutput(global_step=1957, training_loss=0.012960956937799542, metrics={'train_runtime': 1038.9846, 'train_samples_per_second': 7.534, 'train_steps_per_second': 1.884, 'total_flos': 1.7503827675788083e+17, 'train_loss': 0.012960956937799542, 'epoch': 1.0})

In [14]:
from src.scripts.utils import predict_entities_batch

texts = [
    # "O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula, com base na Lei 17.681/2017.",
    "- Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica ."
]

results = predict_entities_batch(texts, model, t.tokenizer)
for res in results:
    print(res)

Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto
- Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica .
Resposta:
-:OA
 Tratando-se:OA
 de:OA
 ação:OA
 indenizatória:OA
 ajuizada:OA
 por:OA
 pessoa:OA
 inc

In [7]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1390
    })
})

In [10]:
data['train']['input_ids'][0]

[69286,
 3958,
 4443,
 32297,
 9087,
 16037,
 86020,
 3955,
 141402,
 4154,
 3524,
 23244,
 1197,
 13596,
 976,
 1467,
 436,
 13,
 1789,
 2121,
 1197,
 13596,
 1709,
 24709,
 36144,
 3524,
 23244,
 29610,
 1447,
 12,
 2726,
 58487,
 2843,
 135734,
 25,
 8550,
 485,
 7806,
 264,
 1197,
 13596,
 1709,
 4009,
 309,
 2872,
 17090,
 15249,
 11,
 7953,
 50304,
 11,
 7759,
 1963,
 15249,
 2569,
 2838,
 2782,
 11,
 506,
 6140,
 82,
 11,
 4992,
 624,
 12,
 393,
 9996,
 41339,
 25,
 6982,
 64,
 1197,
 13596,
 1709,
 29610,
 9662,
 288,
 409,
 45962,
 64846,
 15185,
 624,
 12,
 75670,
 2045,
 25,
 2876,
 924,
 1197,
 13596,
 1709,
 3158,
 309,
 64066,
 18965,
 2782,
 11,
 7953,
 16879,
 11,
 4812,
 37085,
 11,
 76082,
 16385,
 11,
 4992,
 624,
 12,
 42501,
 25,
 2263,
 3001,
 1197,
 13596,
 1709,
 4009,
 309,
 92824,
 3893,
 82481,
 16627,
 11,
 7953,
 272,
 13596,
 11,
 70877,
 11,
 93524,
 11,
 835,
 485,
 47919,
 11,
 4992,
 624,
 12,
 35426,
 1637,
 17845,
 74634,
 25,
 22507,
 29488,
 1197,


In [12]:
t.tokenizer.decode(
    data['train']['input_ids'][2], 
    add_special_tokens=True
)

'Você é um especialista jurídico responsável por identificar entidades em textos.        \nAs entidades que você deve identificar são:\n\n- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.\n- PESSOA: Designa entidades que são nomes de pessoas físicas.\n- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.\n- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.\n- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.\n- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      \n\nsegue o texto\nTexto: - Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica .\nResposta:\n Entidades:ORGANIZACAO: Ministério Público<|im_end|><|im_end|><|im_end